In [1]:
!pip install groq --q
print("Installed Succesfully..!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.3 MB/s eta 0:00:00
Installed Succesfully..!


In [2]:
import sqlite3
import pandas as pd
import os

from groq import Groq
import re

print("All Libraries Installed Succesfully..!")

All Libraries Installed Succesfully..!


In [4]:
os.environ["GROQ_API_KEY"] = "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"
client = Groq(api_key=os.environ["GROQ_API_KEY"])

model = "llama-3.1-8b-instant"

print("Groq client initialized successfully")
print(f"Using model: {model}")

Groq client initialized successfully
Using model: llama-3.1-8b-instant


In [5]:
import io

csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""

df = pd.read_csv(io.StringIO(csv_data))

print(f"Dataset loaded : {len(df)} rows, {len(df.columns)} columns")
print("\nFirst 5 rows:")
df.head()

Dataset loaded : 30 rows, 8 columns

First 5 rows:


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [6]:
conn = sqlite3.connect("college_db")
df.to_sql("students", conn, if_exists="replace", index=False)

test_df = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students", conn)
print(f"\nVerification: {test_df['total_rows'][0]} rows in database")


Verification: 30 rows in database


In [7]:
def get_schema(conn, table_name="students"):
  cursor = conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns = cursor.fetchall()

  schema_lines = [f"Table: {table_name}"]
  schema_lines.append("Columns:")

  for col in columns:
    schema_lines.append(f"  - {col[1]} ({col[2]})")

  cursor.execute(f"SELECT * FROM {table_name} LIMIT 1")
  sample_rows = cursor.fetchall()

  for row in sample_rows:
    schema_lines.append(f" {row}")

  return "\n".join(schema_lines)
schema = get_schema(conn)
print(schema)

Table: students
Columns:
  - student_id (INTEGER)
  - name (TEXT)
  - age (INTEGER)
  - gender (TEXT)
  - subject (TEXT)
  - marks (INTEGER)
  - attendance (INTEGER)
  - grade (TEXT)
 (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')


In [8]:
def generate_sql(user_question,schema_text,client,model):
  system_prompt = f"""You are an expert SQL assistant.
  You are connected to a SQLite database with the following structure:
    {schema_text}
      Rules you must follow:
    1. Generate ONLY a valid SQLite SQL query.
    2. Do not include any explanation or text — only the SQL query.
    3. Do not use markdown code blocks. Return the raw SQL only.
    4. The table name is: students
    5. Only use column names that exist in the schema above.
    6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Programming').
    7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
    """
  response=client.chat.completions.create(
      model=model,
      messages=[
          {"role":"system","content":system_prompt},
          {"role":"user","content":user_question}
      ],
      temperature=0
  )
  sql_query=response.choices[0].message.content.strip()
  return sql_query
question="Show me all female students"
print(f"{question}")
sql=generate_sql(question,schema,client,model)
print(sql)

Show me all female students
SELECT * FROM students WHERE gender = 'Female'


In [9]:
def execute_sql(sql_query,conn):
  clean_sql=sql_query.strip()
  clean_sql=re.sub(r'```sql\s*','',clean_sql)
  clean_sql=re.sub(r'```\s*','',clean_sql)
  clean_sql=clean_sql.strip()

  try:
    result_df=pd.read_sql_query(clean_sql,conn)
    return result_df,None
  except Exception as e:
    return None,str(e)

result,error=execute_sql(sql,conn)
if error:
  print(error)
else:
  print(result)

    student_id            name  age  gender      subject  marks  attendance  \
0            2     Priya Patel   21  Female      Science     76          85   
1            4      Sneha Iyer   22  Female  Mathematics     62          78   
2            6  Divya Krishnan   20  Female      Science     83          88   
3            8    Ananya Gupta   21  Female  Programming     89          96   
4           10    Pooja Sharma   22  Female  Mathematics     55          72   
5           12   Meera Nambiar   20  Female      Science     81          87   
6           14   Kavitha Rajan   21  Female  Programming     86          93   
7           16   Swathi Pillai   22  Female  Mathematics     90          95   
8           18   Lavanya Menon   20  Female      Science     66          76   
9           20    Anjali Singh   21  Female  Programming     94          97   
10          22    Rekha Sharma   22  Female  Mathematics     58          73   
11          24       Usha Iyer   20  Female      Sci

In [10]:
question1="Show me the first 5 female students"
sql1=generate_sql(question1,schema,client,model)
print(f"Executing SQL: {sql1}")
result1, error1 = execute_sql(sql1,conn)

if error1:
  print(f"Error : {error1}")
else:
  print(f"\nQuery returned {len(result1)} rows")
  print(result1)

Executing SQL: SELECT * FROM students WHERE gender = 'Female' ORDER BY student_id ASC LIMIT 5

Query returned 5 rows
   student_id            name  age  gender      subject  marks  attendance  \
0           2     Priya Patel   21  Female      Science     76          85   
1           4      Sneha Iyer   22  Female  Mathematics     62          78   
2           6  Divya Krishnan   20  Female      Science     83          88   
3           8    Ananya Gupta   21  Female  Programming     89          96   
4          10    Pooja Sharma   22  Female  Mathematics     55          72   

  grade  
0     B  
1     C  
2     A  
3     A  
4     D  


In [17]:
def text_to_sql_agent(user_question,conn,client,model,verbose=True):

  print("="*60)
  print(f"USER QUESTION: {user_question}")
  print("="*60)

  if verbose:
    print("\n[STEP 1] Reading database schema...")

  schema_text=get_schema(conn)

  if verbose:
    print("Schema loaded sucesfully")

  if verbose:
    print("[step2]Generating Sql query with groq llm")

  generated_sql = generate_sql(user_question, schema_text, client, model)

  if verbose:
    print("SQL generated successfully")
    print(f"\nGenerated SQL: {generated_sql}")

  if verbose:
    print("[step3]Executed sql on the datbase...")

  result_df, error = execute_sql(generated_sql, conn)

  if error:
    print(f"SQL Execution Error: {error}")
    return None, generated_sql

  if verbose:
    print(f"[STEP 4] Query returned {len(result_df)} row(s)")
    print("RESULTS")
    print("-"*40)
    print(result_df.to_string(index=False))

  print("="*60)

  return result_df, generated_sql

result,sql_used = text_to_sql_agent(
    "Show me all female students",
    conn,client,model
)

USER QUESTION: Show me all female students

[STEP 1] Reading database schema...
Schema loaded sucesfully
[step2]Generating Sql query with groq llm
SQL generated successfully

Generated SQL: SELECT * FROM students WHERE gender = 'Female'
[step3]Executed sql on the datbase...
[STEP 4] Query returned 15 row(s)
RESULTS
----------------------------------------
 student_id           name  age gender     subject  marks  attendance grade
          2    Priya Patel   21 Female     Science     76          85     B
          4     Sneha Iyer   22 Female Mathematics     62          78     C
          6 Divya Krishnan   20 Female     Science     83          88     A
          8   Ananya Gupta   21 Female Programming     89          96     A
         10   Pooja Sharma   22 Female Mathematics     55          72     D
         12  Meera Nambiar   20 Female     Science     81          87     A
         14  Kavitha Rajan   21 Female Programming     86          93     A
         16  Swathi Pillai   22 Fe

In [14]:
result1 , _ = text_to_sql_agent("Show me all students who study mathematics",conn,client,model)

USER QUSTION:Show me all students who study mathematics
[Step1]Reading databse schema
Schema loaded sucesfully
[step2]Genearting Sql query with groq llm
Sql generated successfully
[step3]Executed sql on the datbse...

[step4] Query returned 10 rows

RESULTS

 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 Female Mathematics     58          73     D
         25   Vijay Kumar   22   Male Mathematics     71          

In [20]:
result2 , _ = text_to_sql_agent("show me all the students by their age more than 20",conn,client,model)


USER QUESTION: show me all the students by their age more than 20

[STEP 1] Reading database schema...
Schema loaded sucesfully
[step2]Generating Sql query with groq llm
SQL generated successfully

Generated SQL: SELECT * FROM students WHERE age > 20
[step3]Executed sql on the datbase...
[STEP 4] Query returned 19 row(s)
RESULTS
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          2   Priya Patel   21 Female     Science     76          85     B
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          5    Arjun Nair   21   Male Programming     91          94    A+
          7   Karan Singh   22   Male Mathematics     74          81     B
          8  Ananya Gupta   21 Female Programming     89          96     A
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         11  Aditya Kumar   21   Male Programming     97          99    A+
         13   Rahul Desai   

In [21]:
result2 , _ = text_to_sql_agent("give me the count of all the students whos subject is math and their mark",conn,client,model)


USER QUESTION: give me the count of all the students whos subject is math and their mark

[STEP 1] Reading database schema...
Schema loaded sucesfully
[step2]Generating Sql query with groq llm
SQL generated successfully

Generated SQL: SELECT COUNT(student_id), marks FROM students WHERE subject = 'Mathematics'
[step3]Executed sql on the datbase...
[STEP 4] Query returned 1 row(s)
RESULTS
----------------------------------------
 COUNT(student_id)  marks
                10     88


In [22]:
result3 , _ = text_to_sql_agent("show me the student name who got A+ ",conn,client,model)


USER QUESTION: show me the student name who got A+ 

[STEP 1] Reading database schema...
Schema loaded sucesfully
[step2]Generating Sql query with groq llm
SQL generated successfully

Generated SQL: SELECT name FROM students WHERE grade = 'A+'
[step3]Executed sql on the datbase...
[STEP 4] Query returned 7 row(s)
RESULTS
----------------------------------------
         name
  Rohan Mehta
   Arjun Nair
 Aditya Kumar
Swathi Pillai
 Anjali Singh
  Nandita Rao
 Bhavna Mehta


In [25]:
result2 , _ = text_to_sql_agent("enakku ella kanaku students oda per ah kudu",conn,client,model)


USER QUESTION: enakku ella kanaku students oda per ah kudu

[STEP 1] Reading database schema...
Schema loaded sucesfully
[step2]Generating Sql query with groq llm
SQL generated successfully

Generated SQL: SELECT * FROM students WHERE subject = 'Mathematics'
[step3]Executed sql on the datbase...
[STEP 4] Query returned 10 row(s)
RESULTS
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha S